In [2]:
import numpy as np
import matplotlib.pyplot as plt
import mne
import pandas as pd
from pathlib import Path

Fs = 500
DATA_FOLDER = Path("data/calculia")

# Channel names
CH_ANT_FRONTAL = ["Fp1", "Fp2"]
CH_FRONTAL = ["F3", "F4", "Fz", "F7", "F8"]
CH_CENTRAL = ["C3", "C4", "Cz"]
CH_PARIETAL = ["P3", "P4", "Pz"]
CH_OCCIPITAL = ["O1", "O2"]
CH_TEMPORAL = ["T3", "T4", "T5", "T6"]

CH_TOTAL = CH_ANT_FRONTAL + CH_FRONTAL + CH_CENTRAL + CH_PARIETAL + CH_OCCIPITAL + CH_TEMPORAL

In [3]:
from _functions import PCE

PCEs = []

for sbj_id in range(36):
    sbj = f"Subject{sbj_id:02d}"
    rest = {"Subject" : sbj, "Rest" : True, "PCE" : -1}
    calc = {"Subject" : sbj, "Rest" : False, "PCE" : -1}
        
    rest_raw = mne.io.read_raw_edf(DATA_FOLDER / f"{sbj}_1.edf", verbose=False)
    rest_np = np.array(rest_raw[["EEG " + ch for ch in CH_TOTAL]][0])
    rest["PCE"], _ = PCE(rest_np)  
    PCEs.append(rest)

    calc_raw = mne.io.read_raw_edf(DATA_FOLDER / f"Subject{sbj_id:02d}_2.edf", verbose=False)
    calc_np = np.array(calc_raw[["EEG " + ch for ch in CH_TOTAL]][0])
    calc["PCE"], _ = PCE(calc_np)
    PCEs.append(calc)

df_results = pd.DataFrame(PCEs)
df_results.head()

,Subject,Rest,PCE
0,Subject00,True,0.588425
1,Subject00,False,0.586203
2,Subject01,True,0.493260
3,Subject01,False,0.564419
4,Subject02,True,0.585285


## Merge user Information

In [13]:
user_df = pd.read_csv(DATA_FOLDER / "subject-info.csv")
df = df_results.merge(user_df, on = "Subject")
df["Male"] = df.Gender.map({"M" : 1, "F" : 0})
df = df.drop(columns=["Recording year", "Number of subtractions", "Gender"])
df.describe()

,PCE,Age,Count quality,Male
count,72.000000,72.000000,72.000000,72.000000
mean,0.569710,18.250000,0.722222,0.250000
std,0.061105,2.154262,0.451046,0.436051
min,0.400857,16.000000,0.000000,0.000000
25%,0.528295,17.000000,0.000000,0.000000
50%,0.568107,17.000000,1.000000,0.000000
75%,0.598562,19.000000,1.000000,0.250000
max,0.753362,26.000000,1.000000,1.000000
